In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata_cospar = ad.read_h5ad("./data/larry/cospar_tmap_result.h5ad")
print(adata)
print(adata_cospar)

In [ ]:
import numpy as np

def match_by_X_emb(adata, adata_cospar):
    X1 = adata.obsm["X_emb"]
    X2 = adata_cospar.obsm["X_emb"]

    # Convert rows to tuples (hashable, exact float match)
    rows1 = [tuple(r) for r in X1]
    rows2 = [tuple(r) for r in X2]

    # Build lookup: embedding -> index in adata_cospar
    lookup = {row: j for j, row in enumerate(rows2)}

    # Map each row in adata → adata_cospar index (or -1 if missing)
    mapA2B = np.array([lookup.get(row, -1) for row in rows1], dtype=int)

    return mapA2B

mapA2B = match_by_X_emb(adata, adata_cospar)
print("Matched:", np.sum(mapA2B != -1), "/", len(mapA2B))

In [ ]:
from scipy.spatial import cKDTree
import joblib

# ===============================================================
# 0. Load embedder
# ===============================================================
emb = joblib.load("./data/larry/larry_embedder.pkl")
emb.gene_names = adata.var_names
emb.cospar_index = mapA2B     # your mapping (adata → adata_cospar)


# ===============================================================
# 1. Tentatively assign new_X_emb from FlowMap
# ===============================================================
n_cospar = adata_cospar.n_obs
new_X_emb = np.full((n_cospar, 2), np.nan)   # initialize with NaNs

idx = emb.cospar_index
mask = idx != -1                             # valid matches only

# fill matched rows
new_X_emb[idx[mask]] = emb.X_emb[mask]


# ===============================================================
# 2. Identify rows with NaN in new_X_emb
# ===============================================================
nan_rows = np.where(~np.isfinite(new_X_emb).all(axis=1))[0]
print("Cells needing patching:", len(nan_rows))


# ===============================================================
# 3. Find nearest neighbors in *original CoSpar embedding*
# ===============================================================
X_cospar_orig = adata_cospar.obsm["X_emb"]    # THIS is the reference geometry

# Use only rows whose new_X_emb is NOT NaN
valid_rows = np.where(np.isfinite(new_X_emb).all(axis=1))[0]

# KD-tree in original embedding space, but only valid rows
tree = cKDTree(X_cospar_orig[valid_rows])

# query NN for each NaN row
_, nn_idx = tree.query(X_cospar_orig[nan_rows], k=1)

# convert relative NN indices → global indices
global_nn_idx = valid_rows[nn_idx]


# ===============================================================
# 4. Replace NaN rows in new_X_emb with the NN's embedding
# ===============================================================
new_X_emb[nan_rows] = new_X_emb[global_nn_idx]


# ===============================================================
# 5. Store the patched embedding back into CoSpar object
# ===============================================================
adata_cospar.obsm["X_emb"] = new_X_emb

print("DONE: adata_cospar.X_emb refreshed and patched.")

In [ ]:
import matplotlib.pyplot as plt
from scripts.FlowCurvature import compute_flow_curvature
from scripts.plotting import *

# ==============================================================
# 0. Compute curvature + acceleration decomposition
# ==============================================================
X_emb = adata_cospar.obsm["X_emb"]
curv = compute_flow_curvature(emb, X_emb)

A        = curv["A"]          # (N, D)
A_tan    = curv["A_tan"]      # (N, d)
A_along  = curv["A_along"]    # (N, D)
A_steer  = curv["A_steer"]    # (N, d)
A_nor    = curv["A_nor"]      # (N, D)

# Raw magnitudes
A_norm        = np.linalg.norm(A, axis=1)
A_tan_norm    = np.linalg.norm(A_tan, axis=1)
A_steer_norm  = np.linalg.norm(A_steer, axis=1)
A_nor_norm    = np.linalg.norm(A_nor,   axis=1)

# ==============================================================
# 1. Quantile clipping helper
# ==============================================================

def clip_quantile(arr, q_low=2, q_high=98):
    lo, hi = np.percentile(arr, [q_low, q_high])
    return np.clip(arr, lo, hi)

speed  = np.linalg.norm(curv["V"], axis=1) + 1e-12
speed2 = speed**2

# curvature-scaled magnitudes
A_curv_norm       = A_norm       / speed2
A_tan_curv_norm   = A_tan_norm   / speed2
A_steer_curv_norm = A_steer_norm / speed2
A_nor_curv_norm   = A_nor_norm   / speed2

# clipped for visualization
A_curv_c       = clip_quantile(A_curv_norm)
A_tan_curv_c   = clip_quantile(A_tan_curv_norm)
A_steer_curv_c = clip_quantile(A_steer_curv_norm)
A_nor_curv_c   = clip_quantile(A_nor_curv_norm)

# ==============================================================
# 2. Plot curvature-scaled components
# ==============================================================

common_kwargs = dict(
    tps_vf=emb.tps_vf,
    stream_density=0.9,
    streamline_thickness=5.0,
    arrowsize=1.5,
    scatter_size=8,
    scatter_alpha=0.6,
    cmap="coolwarm",
    show_axes=False
)

# --- total acceleration curvature ---
plot_velocity_streamplot(
    X_emb,
    scatter_color=A_curv_c,
    **common_kwargs
)

# --- tangent acceleration curvature ---
plot_velocity_streamplot(
    X_emb,
    scatter_color=A_tan_curv_c,
    **common_kwargs
)

# --- steering (geodesic) curvature contribution ---
plot_velocity_streamplot(
    X_emb,
    scatter_color=A_steer_curv_c,
    **common_kwargs
)

# --- normal (extrinsic) curvature contribution ---
plot_velocity_streamplot(
    X_emb,
    scatter_color=A_nor_curv_c,
    **common_kwargs
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scripts.FlowCurvature import compute_flow_curvature
from scripts.plotting import *

# ==============================================================
# 0. Compute curvature + acceleration decomposition
# ==============================================================

curv = compute_flow_curvature(emb, emb.X_emb)

A        = curv["A"]          # (N, D)
A_tan    = curv["A_tan"]      # (N, d)
A_along  = curv["A_along"]    # (N, D)
A_steer  = curv["A_steer"]    # (N, d)
A_nor    = curv["A_nor"]      # (N, D)

# Raw magnitudes
A_norm        = np.linalg.norm(A, axis=1)
A_tan_norm    = np.linalg.norm(A_tan, axis=1)
A_steer_norm  = np.linalg.norm(A_steer, axis=1)
A_nor_norm    = np.linalg.norm(A_nor,   axis=1)

# ==============================================================
# 1. Quantile clipping helper
# ==============================================================

def clip_quantile(arr, q_low=2, q_high=98):
    lo, hi = np.percentile(arr, [q_low, q_high])
    return np.clip(arr, lo, hi)

speed  = np.linalg.norm(curv["V"], axis=1) + 1e-12
speed2 = speed**2

# curvature-scaled magnitudes
A_curv_norm       = A_norm       / speed2
A_tan_curv_norm   = A_tan_norm   / speed2
A_steer_curv_norm = A_steer_norm / speed2
A_nor_curv_norm   = A_nor_norm   / speed2

# clipped for visualization
A_curv_c       = clip_quantile(A_curv_norm)
A_tan_curv_c   = clip_quantile(A_tan_curv_norm)
A_steer_curv_c = clip_quantile(A_steer_curv_norm)
A_nor_curv_c   = clip_quantile(A_nor_curv_norm)

# ==============================================================
# 2. Plot curvature-scaled components
# ==============================================================

common_kwargs = dict(
    tps_vf=emb.tps_vf,
    stream_density=0.9,
    streamline_thickness=5.0,
    arrowsize=1.5,
    scatter_size=8,
    scatter_alpha=0.6,
    cmap="coolwarm",
    show_axes=False
)

# --- total acceleration curvature ---
plot_velocity_streamplot(
    emb.X_emb,
    scatter_color=A_curv_c,
    **common_kwargs
)

# --- tangent acceleration curvature ---
plot_velocity_streamplot(
    emb.X_emb,
    scatter_color=A_tan_curv_c,
    **common_kwargs
)

# --- steering (geodesic) curvature contribution ---
plot_velocity_streamplot(
    emb.X_emb,
    scatter_color=A_steer_curv_c,
    **common_kwargs
)

# --- normal (extrinsic) curvature contribution ---
plot_velocity_streamplot(
    emb.X_emb,
    scatter_color=A_nor_curv_c,
    **common_kwargs
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

from scripts.plotting import compute_velocity_on_grid


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def points_inside_mask(X, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X)
    r = np.median(nn.kneighbors(X)[0][:, -1]) * radius_scale
    neigh = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(ix) > 0 for ix in neigh])


# -----------------------------------------------------------------------------
# Data
# -----------------------------------------------------------------------------
curvature = A_steer_curv_c
V_cells = emb.tps_vf.predict(X_emb)


# -----------------------------------------------------------------------------
# Velocity grid (sparse but visible)
# -----------------------------------------------------------------------------
Xg, _, _ = compute_velocity_on_grid(
    X_emb,
    grid_size=25,
    min_mass=0.02
)

keep = points_inside_mask(X_emb, Xg)
Xg = Xg[keep][::2]
Vg = emb.tps_vf.predict(Xg)


# -----------------------------------------------------------------------------
# Plot
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6))

# Cells (main signal)
ax.scatter(
    X_emb[:, 0], X_emb[:, 1],
    c=curvature,
    cmap="coolwarm",
    s=20,              # ↓ smaller dots
    alpha=0.15,
    linewidths=0,
    zorder=1
)

# Velocity field (clear but not dominant)
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy",
    scale_units="xy",
    scale=4.0,
    width=0.0045,      # ↑ thicker shaft (this is the key knob)
    color="k",
    alpha=0.7,         # ↑ a bit darker for print
    headwidth=4.2,
    headlength=3.8,
    headaxislength=3.2,
    zorder=2
)


# -----------------------------------------------------------------------------
# Formatting
# -----------------------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.colors as mcolors
# import mpltern

# # ==============================================================
# # 0. Normalize A by speed**2 (curvature-consistent)
# # ==============================================================

# V = curv["V"]
# speed = np.linalg.norm(V, axis=1) + 1e-12
# speed2 = speed**2 + 1e-12

# A_rate       = curv["A"]       / speed2[:, None]
# A_along_rate = curv["A_along"] / speed2[:, None]
# A_steer_rate = curv["A_steer"] / speed2[:, None]
# A_nor_rate   = curv["A_nor"]   / speed2[:, None]

# # ==============================================================
# # 1. Magnitudes and fractions
# # ==============================================================

# mag_along = np.linalg.norm(A_along_rate, axis=1)
# mag_steer = np.linalg.norm(A_steer_rate, axis=1)
# mag_nor   = np.linalg.norm(A_nor_rate,   axis=1)

# mag_total = mag_along + mag_steer + mag_nor + 1e-12

# f_along = mag_along / mag_total
# f_steer = mag_steer / mag_total
# f_nor   = mag_nor   / mag_total

# # Ternary coordinates
# t = f_steer * 100
# l = f_nor   * 100
# r = f_along * 100

# # ==============================================================
# # 2. Color map
# # ==============================================================

# labels = np.asarray(adata.obs["state_info"].values)
# uniq = np.unique(labels)
# other = [lab for lab in uniq if lab != "Undifferentiated"]

# cmap = plt.get_cmap("tab10", len(other))
# colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
# colmap["Undifferentiated"] = "#d3d3d3"

# # ==============================================================
# # 3. Plot
# # ==============================================================

# fig = plt.figure(figsize=(6.3, 6.1))
# ax = fig.add_subplot(1, 1, 1, projection="ternary", ternary_sum=100.0)
# ax.set_facecolor("white")

# # --- Boundary ---
# for spine in ax.spines.values():
#     spine.set_color("black")
#     spine.set_linewidth(3.5)

    
# # --------------------------------------------------------------
# # FIRST: scatter Undifferentiated cells (background)
# # --------------------------------------------------------------
# mask_undiff = labels == "Undifferentiated"
# ax.scatter(
#     t[mask_undiff], l[mask_undiff], r[mask_undiff],
#     color=colmap["Undifferentiated"],
#     s=5,
#     alpha=0.15,
#     edgecolor="none",
#     zorder=1
# )

# # --------------------------------------------------------------
# # SECOND: scatter differentiated cells (foreground)
# # --------------------------------------------------------------
# for lab in uniq:
#     if lab == "Undifferentiated":
#         continue
#     mask = labels == lab
#     ax.scatter(
#         t[mask], l[mask], r[mask],
#         color=colmap[lab],
#         s=5,
#         alpha=0.3,
#         edgecolor="none",
#         zorder=10
#     )    
# # ==============================================================
# # Dense, readable grid
# # ==============================================================
# grid_ticks = [0, 25, 50, 75, 100]

# ax.grid(color="0.7", linewidth=1.4)
# ax.taxis.set_ticks(grid_ticks)
# ax.laxis.set_ticks(grid_ticks)
# ax.raxis.set_ticks(grid_ticks)

# # ==============================================================
# # === AXIS COLOR LINES BLOCK (toggle ON for exploration) =======

# for v in [25, 50, 75]:
#     ax.axtline(v, color='C0', linewidth=1.8, alpha=0.7)
#     ax.axlline(v, color='C1', linewidth=1.8, alpha=0.7)
#     ax.axrline(v, color='C2', linewidth=1.8, alpha=0.7)

# ax.set_tlabel("steer")
# ax.set_llabel("normal")
# ax.set_rlabel("along")

# ax.taxis.set_ticklabels([f"{v}%" for v in grid_ticks], fontsize=13, color='C0')
# ax.laxis.set_ticklabels([f"{v}%" for v in grid_ticks], fontsize=13, color='C1')
# ax.raxis.set_ticklabels([f"{v}%" for v in grid_ticks], fontsize=13, color='C2')

# # ==============================================================

# # --- Clean version: keep colored lines but remove labels ---
# # for v in [25, 50, 75]:
# #     ax.axtline(v, color='C0', linewidth=2.8, alpha=0.7)
# #     ax.axlline(v, color='C1', linewidth=2.8, alpha=0.7)
# #     ax.axrline(v, color='C2', linewidth=2.8, alpha=0.7)

# # ax.taxis.set_ticklabels([])
# # ax.laxis.set_ticklabels([])
# # ax.raxis.set_ticklabels([])
# # ax.set_tlabel("")
# # ax.set_llabel("")
# # ax.set_rlabel("")


# plt.show()

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def grow_clusters(coords, vals, seed_points, k=5, m=0.12):
    """
    Dumb region-growing clusterer on the FULL embedding.
    coords: (N,2)      full embedding
    vals:   (N,)       full curvature/intensity array
    seed_points: (S,2) seed locations in SAME embedding space
    k: neighbors for KNN
    m: threshold (grow into vals > m)
    
    Returns:
        seed_idx_global: indices in full coords
        labels_full:     cluster labels for all N cells
    """
    N = coords.shape[0]

    # --- (1) map seed points to nearest cells in full coords ---
    seed_idx = []
    for p in seed_points:
        d = np.linalg.norm(coords - p, axis=1)
        seed_idx.append(np.argmin(d))
    seed_idx = np.array(seed_idx)

    # --- (2) Build global KNN ---
    nn = NearestNeighbors(n_neighbors=k+1).fit(coords)
    nbr_indices = nn.kneighbors(return_distance=False)[:, 1:]

    # --- (3) Region growing on full data ---
    labels = -np.ones(N, dtype=int)
    stack = []

    for cid, s in enumerate(seed_idx):
        labels[s] = cid
        stack.append(s)

    while stack:
        i = stack.pop()
        cid = labels[i]

        for j in nbr_indices[i]:
            if labels[j] != -1:
                continue
            if vals[j] > m:    # ✔ matching your original logic
                labels[j] = cid
                stack.append(j)

    return seed_idx, labels

In [ ]:
# ---------------------------------------------------------------------
# Base data
# ---------------------------------------------------------------------
coords = X_emb                    # (N, 2)
vals   = A_steer_curv_c               # (N,)

# ---------------------------------------------------------------------
# Zoom window (same as before)
# ---------------------------------------------------------------------
x_min, x_max = 5.0, 10.0
y_min, y_max = 3.5, 8.0

mask_zoom = (
    (coords[:, 0] > x_min) & (coords[:, 0] < x_max) &
    (coords[:, 1] > y_min) & (coords[:, 1] < y_max)
)

# ---------------------------------------------------------------------
# Zoomed data
# ---------------------------------------------------------------------
coords_zoom = coords[mask_zoom]
vals_zoom   = vals[mask_zoom]

# ---------------------------------------------------------------------
# Seed points for cluster growth
# ---------------------------------------------------------------------
seed_points = np.array([
    [7.5, 6.0],
    [7., 4.5],
])

# ---------------------------------------------------------------------
# Run cluster growth on full data
# ---------------------------------------------------------------------
seed_idx_full, labels_zoom = grow_clusters(
    coords_zoom,
    vals_zoom,
    seed_points,
    k=7,
    m=0.09
)


fig, ax = plt.subplots(figsize=(6, 6))

mask_assigned = labels_zoom != -1

# Unassigned cells
ax.scatter(
    coords_zoom[~mask_assigned, 0],
    coords_zoom[~mask_assigned, 1],
    c="lightgray",
    s=15,
    alpha=0.3,
    linewidths=0,
    zorder=1
)

# Assigned clusters
ax.scatter(
    coords_zoom[mask_assigned, 0],
    coords_zoom[mask_assigned, 1],
    c=labels_zoom[mask_assigned],
    cmap="tab10",
    s=30,
    alpha=0.85,
    linewidths=0,
    zorder=2
)

# --------------------------------------------------
# Seed markers (comment out this block to remove)
# --------------------------------------------------
# ax.scatter(
#     seed_points[:, 0],
#     seed_points[:, 1],
#     marker="x",
#     s=160,
#     c="black",
#     linewidths=2.5,
#     zorder=10
# )

ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

sc = ax.scatter(
    coords_zoom[:, 0],
    coords_zoom[:, 1],
    c=vals_zoom,
    cmap="coolwarm",
    s=160,
    alpha=0.35,
    linewidths=0,
    zorder=1
)

ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# subset COSPAR AnnData
adata_cospar_zoom = adata_cospar[mask_zoom].copy()
adata_cospar_zoom.obs["cluster"] = labels_zoom.astype(str)

adata_cospar_zoom.raw = adata_cospar_zoom
adata_cospar_zoom.obsm["X_umap"] = adata_cospar_zoom.obsm["X_emb"]

# differential expression
cluster_A, cluster_B = "0", "1"

sc.tl.rank_genes_groups(
    adata_cospar_zoom,
    groupby="cluster",
    groups=[cluster_A],
    reference=cluster_B,
    method="wilcoxon",
)

print(adata_cospar_zoom.obs["cluster"].value_counts(), "\n")

# collect DE results
de = adata_cospar_zoom.uns["rank_genes_groups"]

deg_df = (
    pd.DataFrame({
        "gene": de["names"][cluster_A],
        "score": de["scores"][cluster_A],
        "logFC": de["logfoldchanges"][cluster_A],
        "pval": de["pvals"][cluster_A],
        "pval_adj": de["pvals_adj"][cluster_A],
    })
    .dropna()
    .sort_values("pval_adj")
)

print(deg_df.head(20))

In [ ]:
alpha = 0.05
logfc_cut = 1.0

# significance flags
df_clean = deg_df.dropna(subset=["logFC", "pval_adj"]).copy()
df_clean["-log10p"] = -np.log10(df_clean["pval_adj"].clip(lower=1e-300))

sig_both = (df_clean["pval_adj"] < alpha) & (df_clean["logFC"].abs() >= logfc_cut)

# pick top genes (annotation optional)
N_annotate = 7
df_sig_sorted = (
    df_clean.loc[sig_both]
             .sort_values("pval_adj")
             .head(N_annotate)
)

# ------------------------------------------------
# PLOT
# ------------------------------------------------
plt.figure(figsize=(6., 7.8))

# background
plt.scatter(
    df_clean.loc[~sig_both, "logFC"],
    df_clean.loc[~sig_both, "-log10p"],
    s=140, c="#C7C7C7", alpha=0.55, edgecolors="none"
)

# significant points
plt.scatter(
    df_clean.loc[sig_both, "logFC"],
    df_clean.loc[sig_both, "-log10p"],
    s=240, c="#B22222", alpha=0.9,
    edgecolors="black", linewidth=0.25
)

# cutoff lines
plt.axhline(-np.log10(alpha), color="black", linestyle="--", lw=1)
plt.axvline(logfc_cut, color="black", linestyle="--", lw=1)
plt.axvline(-logfc_cut, color="black", linestyle="--", lw=1)

# ------------------------------------------------
# OPTIONAL MANUAL ANNOTATION SECTION
# ------------------------------------------------
for _, row in df_sig_sorted.iterrows():
    plt.text(
        row["logFC"] + 0.10,
        row["-log10p"] + 0.10,
        row["gene"],
        fontsize=24,
        ha="left",
        va="bottom"
    )

# ------------------------------------------------
# Aesthetics
# ------------------------------------------------
ax = plt.gca()

# keep only axis lines
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["bottom", "left"]:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_linewidth(1.3)

# axis labels
plt.xlabel(r"log$_2$ FC (a / b)", fontsize=36)
plt.ylabel(r"-log$_{10}$(adjusted p)", fontsize=36)

# ------------------------------------------------
# Ticks: increase size & sparsity
# ------------------------------------------------

# y ticks: 0, 30, 60
yticks = [0, 10, 20]
plt.yticks(yticks, [str(y) for y in yticks], fontsize=24)

# x ticks: spaced, larger
plt.xticks([-10, -5, 0, 5, 10], fontsize=24)

plt.xlim(-10, 10)
plt.grid(False)
plt.tick_params(axis="both", length=5, width=1.2, color="black")

plt.tight_layout()
plt.show()

In [ ]:
import cospar as cs

selected_fates = [
    ["Meg", "Erythroid"],
    ["Mast", "Baso"]
]

cs.tl.fate_map(
    adata_cospar,
    selected_fates=selected_fates,
    source="transition_map",
    map_backward=True
)

cs.tl.fate_bias(
    adata_cospar,
    selected_fates=selected_fates,
    source="transition_map",
    pseudo_count=0,
    sum_fate_prob_thresh=0.1,
)

In [ ]:
cs.tl.progenitor(
    adata_cospar,
    selected_fates=selected_fates,
    source="transition_map",
    map_backward=True,
    bias_threshold_A=0.5,
    bias_threshold_B=0.5,
    sum_fate_prob_thresh=0.2,
    avoid_target_states=True,
)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# ---------------------------------------------------
# 0. Configuration
# ---------------------------------------------------
genes = (
    deg_df
    .sort_values("pval_adj")
    .head(25)["gene"]
    .tolist()
)

genes = [g for g in genes if g in adata_cospar.var_names]

bias_key = 'fate_bias_transition_map_Meg_Erythroid*Mast_Baso'
prog_A = "progenitor_transition_map_Meg_Erythroid"
prog_B = "progenitor_transition_map_Mast_Baso"

eps = 1e-6
n_grid = 300          # resolution of continuous axis
sigma = 30             # smoothing strength (increase = smoother)

# ---------------------------------------------------
# 1. Progenitor UNION mask
# ---------------------------------------------------
prog_mask = (
    adata_cospar.obs[prog_A].values.astype(bool)
    | adata_cospar.obs[prog_B].values.astype(bool)
)

# ---------------------------------------------------
# 2. Extract bias + expression
# ---------------------------------------------------
bias = adata_cospar.obs[bias_key].values[prog_mask]

valid = np.abs(bias - 0.5) > eps
bias = bias[valid]

X = adata_cospar[prog_mask, genes].X
if hasattr(X, "A"):
    X = X.A
X = X[valid]

# ---------------------------------------------------
# 3. Sort by fate bias
# ---------------------------------------------------
order = np.argsort(bias)
bias = bias[order]
X = X[order]

# ---------------------------------------------------
# 4. Smooth expression along fate bias
# ---------------------------------------------------
# regular bias grid
bias_grid = np.linspace(0, 1, n_grid)

X_smooth = np.zeros((len(genes), n_grid))

for gi in range(len(genes)):
    # interpolate gene expression onto grid
    X_interp = np.interp(bias_grid, bias, X[:, gi])
    
    # smooth along bias axis
    X_smooth[gi] = gaussian_filter1d(X_interp, sigma=sigma)

# ---------------------------------------------------
# 5. Z-score per gene (row-wise)
# ---------------------------------------------------
mean = X_smooth.mean(axis=1, keepdims=True)
std = X_smooth.std(axis=1, keepdims=True) + 1e-8
Xz = (X_smooth - mean) / std
Xz = np.clip(Xz, -2, 2)

from scipy.cluster.hierarchy import linkage, leaves_list

Z = linkage(Xz, method="average", metric="euclidean")
row_order = leaves_list(Z)

Xz_ord = Xz[row_order]
genes_ord = [genes[i] for i in row_order]


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, ax = plt.subplots(figsize=(5.5, 3.5))

sns.heatmap(
    Xz_ord,
    ax=ax,
    cmap="cividis",
    center=0,
    yticklabels=genes_ord,
    xticklabels=False,
    cbar=False,
    rasterized=True
)

# ---------------------------------------------------
# Fate-bias boundary
# ---------------------------------------------------
boundary_idx = np.argmin(np.abs(bias_grid - 0.5))
ax.axvline(boundary_idx, color="red", linewidth=4)

# ---------------------------------------------------
# X-axis ticks (fate bias)
# ---------------------------------------------------
target_ticks = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
tick_pos = [np.argmin(np.abs(bias_grid - t)) for t in target_ticks]

ax.set_xticks(tick_pos)
ax.set_xticklabels([f"{t:.1f}" for t in target_ticks], fontsize=12)

ax.set_xlabel("Fate bias (Mk/Er vs Ma/Ba)", fontsize=14)
ax.set_ylabel("")

ax.tick_params(axis="y", labelsize=9)

plt.tight_layout()

out_path = "./figures/larry/fate_bias_gene_heatmap_region3.pdf"

fig.savefig(
    out_path,
    dpi=300,              # irrelevant for PDF, critical for PNG
    bbox_inches="tight",
    pad_inches=0.02
)

plt.show()

In [ ]:
cs.tl.fate_map(
    adata_cospar,
    selected_fates=["Neutrophil", "Mast", "Meg", "Monocyte", "Baso",   "Erythroid"],
    source="transition_map",
    map_backward=True,
)

In [ ]:
len(mask_zoom), sum(mask_zoom)

In [ ]:
genes = [
    # "Muc13", "Lgals1", "Lfitm3", "Lghm",
    "Cpa3", "Dlk1", "Cd34", "Gsn"
]

# keep only zoom clusters 0 and 1
clusters = adata_cospar.obs["cluster_zoom"].astype(str).values
mask = (clusters == "0") | (clusters == "1")
adata_sub = adata_cospar[mask]

# extract expression
X = adata_sub.X
if not isinstance(X, np.ndarray):
    X = X.A

clusters_sub = adata_sub.obs["cluster_zoom"].astype(str).values


# ---------------------------------------------------
# Build long-form dataframe for seaborn
# ---------------------------------------------------
df_plot = pd.concat([
    pd.DataFrame({
        "gene": g,
        "expression": X[:, np.where(adata_sub.var_names == g)[0][0]],
        "cluster": clusters_sub
    })
    for g in genes
], axis=0)

# ---------------------------------------------------
# Plot: 2 × 4 violin grid
# ---------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=False)
axes = axes.ravel()

for ax, g in zip(axes, genes):
    sns.violinplot(
        data=df_plot[df_plot["gene"] == g],
        x="cluster",
        y="expression",
        ax=ax,
        cut=0,
        inner="box",
        linewidth=1,
        palette=["#4e79a7", "#f28e2b"],
    )
    ax.set_title(g, fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# --- basic cluster counts (all cells) ---
clusters = adata_cospar.obs["cluster_zoom"].astype(str)

print("All cluster_zoom labels:")
print(clusters.value_counts(dropna=False))

print("\nCluster_zoom 0 vs 1:")
print(clusters.value_counts().reindex(["0", "1"]).fillna(0).astype(int))


# --- counts after subset ---
mask01 = clusters.isin(["0", "1"])
clusters01 = clusters[mask01]

print("\nSubset to cluster_zoom {0,1}:")
print(clusters01.value_counts().reindex(["0", "1"]).fillna(0).astype(int))


# --- OPTIONAL: how many cells have NONZERO fate prob ---
fate = "Erythroid"
key = f"fate_map_transition_map_{fate}"
p = adata_cospar.obs[key].values

eps = 1e-12
nonzero = p > eps

tab = pd.DataFrame({
    "cluster": clusters.values,
    "in01": mask01.values,
    "nonzero": nonzero,
})

print(f"\nNonzero {fate} probs per cluster_zoom (eps={eps}):")
print(
    tab.groupby("cluster")["nonzero"]
       .sum()
       .sort_values(ascending=False)
)

print(f"\nNonzero {fate} probs for cluster_zoom 0 vs 1:")
print(
    tab[tab["cluster"].isin(["0","1"])]
      .groupby("cluster")["nonzero"]
      .sum()
      .reindex(["0","1"])
      .fillna(0)
      .astype(int)
)

print(f"\nTotal cells for cluster_zoom 0 vs 1 (reference):")
print(
    tab[tab["cluster"].isin(["0","1"])]
      .groupby("cluster")
      .size()
      .reindex(["0","1"])
      .fillna(0)
      .astype(int)
)

In [ ]:
from matplotlib.ticker import FuncFormatter
from scipy.stats import mannwhitneyu
import matplotlib.colors as mcolors

def lighten_color(color, amount=0.6):
    c = np.array(mcolors.to_rgb(color))
    return tuple(c + (1 - c) * amount)

def percent_fmt_sparse(x, _):
    p = x * 100
    if abs(p - 0.5) < 1e-6:
        return "0.5%"
    if abs(p - round(p)) < 1e-6:
        return f"{int(round(p))}%"
    return ""

def p_to_stars(p):
    if p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


# ---------------------------------------------------
# Configuration
# ---------------------------------------------------
cluster_map = {"0": "a", "1": "b"}
palette = {"a": "#4e79a7", "b": "#f28e2b"}

# Only erythroid-related fates
fate_grid = [
    ["Mast", "Meg"],
    ["Baso", "Erythroid"],
]


# ---------------------------------------------------
# X-axis ranges per fate (adjust freely)
# ---------------------------------------------------
xlims_fate = {
    "Mast":       (0.0, 0.1),
    "Meg":        (0.0, 0.1),
    "Baso":       (0.0, 0.1),
    "Erythroid":  (0.0, 0.1),
}

xticks = {
    0: [0.00, 0.02, 0.04],
    1: [0.00, 0.005, 0.01],
    2: [0.00, 0.005, 0.01],
}

percent_fmt = FuncFormatter(lambda x, _: f"{round(x * 100, 1)}%")

# ---------------------------------------------------
# 1. Subset + relabel clusters
# ---------------------------------------------------
clusters = adata_cospar.obs["cluster_zoom"].astype(str)
mask = clusters.isin(["0", "1"])
adata_sub = adata_cospar[mask]

# ---------------------------------------------------
# 2. Build long-form dataframe
# ---------------------------------------------------
df_plot = pd.concat([
    pd.DataFrame({
        "fate": fate,
        "probability": adata_sub.obs[f"fate_map_transition_map_{fate}"].values,
        "cluster": adata_sub.obs["cluster_zoom"].map(cluster_map).values,
    })
    for fate in np.ravel(fate_grid)
], axis=0)


# ---------------------------------------------------
# 3. Plot
# ---------------------------------------------------
# ---------------------------------------------------
# Plot
# ---------------------------------------------------
sns.set_theme(style="white", context="paper", font_scale=1.2)

fig, axes = plt.subplots(
    2, 2,
    figsize=(7.5, 6),
    sharey=True,
)

violin_palette = {
    k: lighten_color(v, amount=0.65)
    for k, v in palette.items()
}

for i in range(2):
    for j in range(2):
        fate = fate_grid[i][j]
        ax = axes[i, j]
        df_f = df_plot[df_plot["fate"] == fate]

        sns.violinplot(
            data=df_f,
            x="probability",
            y="cluster",
            ax=ax,
            cut=0,
            inner=None,
            linewidth=0,
            palette=violin_palette,
            orient="h",
            alpha=0.9,
            zorder=1,
        )

        sns.boxplot(
            data=df_f,
            x="probability",
            y="cluster",
            ax=ax,
            width=0.18,
            showcaps=True,
            boxprops={"facecolor": "none", "linewidth": 3.5},
            whiskerprops={"linewidth": 3.5},
            capprops={"linewidth": 3.5},
            medianprops={"color": "black", "linewidth": 5.0},
            showfliers=False,
            orient="h",
            palette=palette,
            zorder=3,
        )

            # ---------------- Stats ----------------
        x_a = df_f.loc[df_f["cluster"] == "a", "probability"].dropna()
        x_b = df_f.loc[df_f["cluster"] == "b", "probability"].dropna()
    
        if len(x_a) >= 3 and len(x_b) >= 3:
            pval = mannwhitneyu(x_a, x_b, alternative="two-sided").pvalue
            stars = p_to_stars(pval)
        else:
            stars = "ns"
    
        # (optional annotation)
        # ax.text(0.1 * 0.95, 0.5, stars,
        #         ha="right", va="center", fontsize=22, fontweight="bold")
        
        ax.set_xlim(*xlims_fate[fate])
        ax.set_title(fate, fontsize=26, pad=8)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(axis="x", labelsize=20)
        ax.tick_params(axis="y", labelsize=22)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()